# E3 — Visualización 3D de reconstrucciones PoinTr

Muestra para cada muestra del test set:
- **Rojo** — nube de puntos rota (entrada al modelo)
- **Verde** — predicción de PoinTr (salida)
- **Azul** — ground truth completo
- **Color por error** — la predicción coloreada por distancia al GT (azul=bien, rojo=mal)

Visualización interactiva con Plotly — rota con el ratón, zoom con la rueda.

**Cambia solo la Celda 3 para ver otro modelo.**

In [ ]:
# ── CELDA 1: Drive ─────────────────────────────────────────────
from google.colab import drive
import os
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print('Drive ya montado.')
else:
    drive.mount('/content/drive')
    print('Drive montado.')

In [ ]:
# ── CELDA 2: Dependencias ──────────────────────────────────────
import os, subprocess
from getpass import getpass

if not os.path.exists('/content/PoinTr'):
    subprocess.run(['git','clone','https://github.com/yuxumin/PoinTr',
                    '/content/PoinTr','--depth=1','-q'], capture_output=True)
    print('PoinTr clonado.')
else:
    print('[OK] PoinTr')

REPO_DIR = '/content/TFM'
if not os.path.exists(REPO_DIR):
    token = getpass('Token GitHub: ')
    subprocess.run(['git','clone',f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D',
                    REPO_DIR,'-q'], capture_output=True)
    del token
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','-q'], capture_output=True)
    print('[OK] TFM repo')

os.chdir(REPO_DIR)
subprocess.run(['git','checkout','raquel/e3','-q'], capture_output=True)
subprocess.run(['pip','install','timm','easydict','pyyaml','plotly','--quiet'])
print('[OK] deps instaladas')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELDA 3 — CONFIGURACION
# ══════════════════════════════════════════════════════════════

# Modelo a visualizar (debe tener best.pt en Drive o local)
VERSION = 'v5_fb_obj'   # opciones: v5_fb_obj, v5_obj, v5_all, v4_pointr_fbv2, v2_pointr...

# Datasets que usó ese modelo (para reconstruir el mismo split)
DATASETS_VERSION = {
    'v5_fb_obj':      ['fb', 'obj'],
    'v5_obj':         ['obj'],
    'v5_all':         ['fb', 'obj', 'sn'],
    'v4_pointr_fbv2': ['fb'],
    'v2_pointr':      ['fb', 'sintetico'],  # aproximado
}
_partes = DATASETS_VERSION.get(VERSION, ['fb', 'obj'])

N_MOSTRAR   = 12    # cuántas muestras del test set visualizar (None = todas)
ALEATORIO   = True  # True = aleatorio, False = primeras N
GUARDAR_HTML = True # guarda cada figura como HTML en E3/visualizaciones/

print(f'Modelo : {VERSION}')
print(f'Partes : {_partes}')
print(f'Mostrar: {N_MOSTRAR} muestras ({"aleatorio" if ALEATORIO else "en orden"})')

In [ ]:
# ── CELDA 4: Cargar modelo + mocks ─────────────────────────────
import sys, os, types, glob as _glob, random
from pathlib import Path

sys.path.insert(0, '/content/PoinTr')
sys.path.insert(0, '/content/TFM')
os.chdir('/content/TFM')

import torch
import torch.nn as nn
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_STR = str(device)
print(f'Dispositivo: {device}')

DRIVE = '/content/drive/MyDrive'
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3/Raquel'
BASE_GENERAL = f'{DRIVE}/Datos_E2_E3/General'

# ── Patch + mocks (igual que en entrenamiento) ───────────────
_pfx = ('models','utils.registry','utils.config','utils.logger','utils.misc','extensions')
_del = [k for k,v in sys.modules.items()
        if any(k==p or k.startswith(p+'.') for p in _pfx)
        or (hasattr(v,'__file__') and v.__file__ and '/content/PoinTr' in str(v.__file__))]
for k in _del: del sys.modules[k]

_np_ = 0
for fp in _glob.glob('/content/PoinTr/models/*.py')+_glob.glob('/content/PoinTr/models/**/*.py'):
    try:
        src=open(fp,encoding='utf-8').read(); new=src.replace('.cuda()',f'.to("{DEVICE_STR}")')
        if new!=src: open(fp,'w',encoding='utf-8').write(new); _np_+=1
    except: pass
print(f'[patch] {_np_} archivos')

def _force(name,attrs):
    m=types.ModuleType(name)
    for k,v in attrs.items(): setattr(m,k,v)
    sys.modules[name]=m
def _inject(name,attrs):
    if name not in sys.modules: _force(name,attrs)
def _cr(a,b): d=torch.cdist(a,b,p=2); return d.min(2).values,d.min(1).values
class _CL1(nn.Module):
    def forward(self,a,b): d1,d2=_cr(a.contiguous(),b.contiguous()); return (d1.mean()+d2.mean())/2
class _CL2(nn.Module):
    def forward(self,a,b): d1,d2=_cr(a.contiguous(),b.contiguous()); return ((d1**2).mean()+(d2**2).mean())/2
class _CPM(nn.Module):
    def forward(self,a,b): return torch.cdist(a.contiguous(),b.contiguous(),p=2).min(2).values.mean()
_ch={'ChamferDistanceL1':_CL1,'ChamferDistanceL2':_CL2,'ChamferDistanceL1_PM':_CPM,'chamfer_3DDist':_cr}
for n in ['chamfer','chamfer_dist','extensions.chamfer_dist','chamfer3D','chamfer3D.dist_chamfer_3D']: _force(n,_ch)

if 'pointnet2_ops' not in sys.modules:
    def _fps(xyz,np__):
        B,N,_=xyz.shape; dev=xyz.device
        idx=torch.zeros(B,np__,dtype=torch.int32,device=dev); dist=torch.full((B,N),1e10,device=dev)
        far=torch.randint(0,N,(B,),dtype=torch.long,device=dev); bi=torch.arange(B,dtype=torch.long,device=dev)
        for i in range(np__):
            idx[:,i]=far.int(); c=xyz[bi,far].unsqueeze(1); dist=torch.min(dist,((xyz-c)**2).sum(-1)); far=dist.max(-1)[1]
        return idx
    def _go(f,idx): B,C,N=f.shape; M=idx.shape[1]; return f.gather(2,idx.long().unsqueeze(1).expand(B,C,M)).contiguous()
    def _bq(r,ns,xyz,nxyz):
        d=torch.cdist(nxyz.float(),xyz.float()); s=d.argsort(-1)[:,:,:ns]
        return torch.where(d.gather(2,s)>r,s[:,:,:1].expand_as(s),s).int()
    def _grp(f,idx):
        B,C,N=f.shape; S,K=idx.shape[1],idx.shape[2]
        return f.gather(2,idx.long().view(B,1,S*K).expand(B,C,S*K)).view(B,C,S,K).contiguous()
    def _3nn(u,k): d=torch.cdist(u.float(),k.float()); d2,i=d.topk(3,-1,largest=False); return d2.float(),i.int()
    def _3i(f,idx,w):
        B,C,M=f.shape; N=idx.shape[1]
        return (f.gather(2,idx.long().view(B,1,N*3).expand(B,C,N*3)).view(B,C,N,3)*w.unsqueeze(1)).sum(-1).contiguous()
    _pu=types.ModuleType('pointnet2_ops.pointnet2_utils')
    for k,v in {'furthest_point_sample':_fps,'gather_operation':_go,'ball_query':_bq,
                'grouping_operation':_grp,'three_nn':_3nn,'three_interpolate':_3i}.items(): setattr(_pu,k,v)
    _pm=types.ModuleType('pointnet2_ops'); _pm.pointnet2_utils=_pu
    sys.modules['pointnet2_ops']=_pm; sys.modules['pointnet2_ops.pointnet2_utils']=_pu

class _KNN(nn.Module):
    def __init__(self,k,transpose_mode=False): super().__init__(); self.k=k; self.tm=transpose_mode
    def forward(self,ref,query):
        if self.tm: d=torch.cdist(query.float(),ref.float()); dk,ik=d.topk(self.k,-1,largest=False); return dk,ik
        r=ref.transpose(1,2).contiguous(); q=query.transpose(1,2).contiguous()
        d=torch.cdist(q.float(),r.float()); dk,ik=d.topk(self.k,-1,largest=False)
        return dk.transpose(1,2),ik.transpose(1,2)
_force('knn_cuda',{'KNN':_KNN})
def _dc(n): return type(n,(nn.Module,),{'__init__':lambda s,*a,**k:super(type(s),s).__init__(),'forward':lambda s,x,*a,**k:x})
class _Emd(nn.Module):
    def forward(self,a,b): return torch.zeros(a.shape[0],device=a.device),torch.zeros(a.shape[0],dtype=torch.int32,device=a.device)
for base,attrs in [('gridding',{'Gridding':_dc('G'),'GriddingReverse':_dc('GR')}),
                   ('gridding_loss',{'GriddingLoss':_dc('GL')}),
                   ('cubic_feature_sampling',{'CubicFeatureSampling':_dc('CFS')}),
                   ('emd',{'emd_module':_Emd,'EarthMoverDistance':_Emd})]:
    for pfx in ['','extensions.']: _inject(pfx+base,attrs)
print('[OK] Mocks')

# ── Cargar checkpoint ────────────────────────────────────────
from easydict import EasyDict
import yaml

ckpt_path = f'E3/checkpoints_pointr_{VERSION}/best.pt'
if not Path(ckpt_path).exists():
    ckpt_path = f'{BASE_E3}/modelos/{VERSION}/best.pt'
    print(f'Cargando desde Drive: {ckpt_path}')

ck = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'Checkpoint: {VERSION} — época {ck["epoch"]}')

try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(EasyDict(ck['model_cfg']))
except:
    from models.PoinTr import PoinTr
    model = PoinTr(EasyDict(ck['model_cfg']))
model.load_state_dict(ck['model_state_dict'])
model = model.to(device); model.eval()
print(f'[OK] Modelo cargado — {sum(p.numel() for p in model.parameters()):,} params')

In [ ]:
# ── CELDA 5: Cargar test set ────────────────────────────────────
import subprocess
from pathlib import Path

FUENTES_TODAS = {
    'fb':  (f'{BASE_GENERAL}/Fantastik_Break_Procesado_v2', 'Datos/fantastic_breaks/procesado_v2'),
    'obj': (f'{BASE_GENERAL}/roturas_Objaverse_v2',          'Datos/objaverse/roturas_v2'),
    'sn':  (f'{BASE_GENERAL}/shapenet_roturas',               'Datos/shapenet/roturas'),
}
FUENTES_ACTIVAS = {k: v for k, v in FUENTES_TODAS.items() if k in _partes}

for clave, (drive_path, local_path) in FUENTES_ACTIVAS.items():
    dst = Path(local_path)
    if dst.exists() and any(dst.glob('*.npy')):
        print(f'[OK] {clave} local: {len(list(dst.glob("*.npy")))} .npy')
    elif Path(drive_path).exists():
        dst.mkdir(parents=True, exist_ok=True)
        print(f'Copiando {clave}...')
        r = subprocess.run(['rsync','-a','--no-links',f'{drive_path}/',str(dst)], capture_output=True, text=True)
        print(f'  {len(list(dst.glob("*.npy")))} .npy copiados')
    else:
        print(f'[WARN] {clave}: no encontrado en Drive ni local')

from E3.dataset import construir_pares, ShapeCompletionDataset
import E3.dataset as _ds
_ds.CENTRAR_EN_ROTO = False

carpetas = [v[1] for v in FUENTES_ACTIVAS.values() if Path(v[1]).exists()]
_todos = construir_pares(carpetas)
_rng = random.Random(42); _rng.shuffle(_todos)
_n = len(_todos); _nt = int(0.8*_n); _nv = int(0.1*_n)
_pares_test = _todos[_nt+_nv:]
print(f'\nTest set: {len(_pares_test)} pares (mismo split que entrenamiento)')

# Seleccionar cuáles mostrar
if ALEATORIO:
    pares_viz = random.sample(_pares_test, min(N_MOSTRAR or len(_pares_test), len(_pares_test)))
else:
    pares_viz = _pares_test[:N_MOSTRAR or len(_pares_test)]
print(f'Mostrando: {len(pares_viz)} muestras')

In [ ]:
# ── CELDA 6: Generar visualizaciones 3D ────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np, torch
from pathlib import Path

vis_dir = Path(f'E3/visualizaciones_{VERSION}')
vis_dir.mkdir(parents=True, exist_ok=True)

def pts_scatter(pts, color, name, size=2, opacity=0.7, colorscale=None, showscale=False):
    """Crea un scatter3d de una nube de puntos."""
    kw = {}
    if colorscale:
        kw['marker'] = dict(size=size, color=color, colorscale=colorscale,
                            showscale=showscale, colorbar=dict(title='Error', len=0.5))
    else:
        kw['marker'] = dict(size=size, color=color, opacity=opacity)
    return go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2], mode='markers',
                        name=name, **kw)

def error_por_punto(pred, gt):
    """Distancia de cada punto predicho al GT más cercano."""
    import torch
    p = torch.tensor(pred).unsqueeze(0)
    g = torch.tensor(gt).unsqueeze(0)
    d = torch.cdist(p, g, p=2)
    return d.min(2).values.squeeze(0).numpy()

def chamfer_np(pred, gt):
    import torch
    p = torch.tensor(pred).unsqueeze(0); g = torch.tensor(gt).unsqueeze(0)
    d = torch.cdist(p, g, p=2)
    return ((d.min(2).values.mean() + d.min(1).values.mean()) / 2).item()

def fscore_np(pred, gt, th=0.01):
    import torch
    p = torch.tensor(pred).unsqueeze(0); g = torch.tensor(gt).unsqueeze(0)
    dpg = torch.cdist(p, g, p=2); dgp = torch.cdist(g, p, p=2)
    pr = (dpg.min(2).values < th).float().mean()
    re = (dgp.min(2).values < th).float().mean()
    if pr + re < 1e-8: return 0.0
    return (2*pr*re/(pr+re)).item()

print(f'Generando {len(pares_viz)} visualizaciones...\n')

for idx_viz, (ruta_r, ruta_c) in enumerate(pares_viz):
    import numpy as np
    roto_np = np.load(ruta_r).astype(np.float32)
    comp_np  = np.load(ruta_c).astype(np.float32)
    nombre = Path(ruta_r).stem.replace('_roto', '')

    # Inferencia
    with torch.no_grad():
        inp = torch.tensor(roto_np).unsqueeze(0).to(device)
        out = model(inp)
        pred_t = out[-1] if isinstance(out, (list, tuple)) else out
        pred_np = pred_t.squeeze(0).cpu().numpy()

    cd  = chamfer_np(pred_np, comp_np)
    fs  = fscore_np(pred_np, comp_np)
    err = error_por_punto(pred_np, comp_np)

    titulo = f'{nombre} | CD={cd:.4f} | F={fs:.4f}'
    print(f'[{idx_viz+1}/{len(pares_viz)}] {titulo}')

    # ── Figura Plotly: 4 paneles ───────────────────────────────
    fig = make_subplots(
        rows=1, cols=4,
        specs=[[{'type':'scatter3d'}]*4],
        subplot_titles=[
            'Entrada (roto)',
            'Predicción PoinTr',
            'Ground Truth (completo)',
            'Error de predicción'
        ]
    )

    # Panel 1: roto (rojo)
    fig.add_trace(pts_scatter(roto_np, '#C62828', 'Roto', size=2), row=1, col=1)

    # Panel 2: predicción (verde)
    fig.add_trace(pts_scatter(pred_np, '#2E7D32', 'Predicción', size=2), row=1, col=2)

    # Panel 3: ground truth (azul)
    fig.add_trace(pts_scatter(comp_np, '#1565C0', 'GT completo', size=2), row=1, col=3)

    # Panel 4: predicción coloreada por error (azul=cerca, rojo=lejos)
    fig.add_trace(
        go.Scatter3d(
            x=pred_np[:,0], y=pred_np[:,1], z=pred_np[:,2],
            mode='markers',
            marker=dict(size=2, color=err, colorscale='RdYlBu_r',
                        showscale=True, colorbar=dict(title='dist', len=0.5, x=1.0)),
            name='Error'
        ),
        row=1, col=4
    )

    # Ejes iguales para los 4 paneles
    escena = dict(
        xaxis=dict(range=[-1.2, 1.2], showticklabels=False, title=''),
        yaxis=dict(range=[-1.2, 1.2], showticklabels=False, title=''),
        zaxis=dict(range=[-1.2, 1.2], showticklabels=False, title=''),
        aspectmode='cube'
    )
    for col in range(1, 5):
        fig.update_layout(**{f'scene{"" if col==1 else col}': escena})

    fig.update_layout(
        title=dict(text=f'<b>{titulo}</b><br><sup>Modelo: PoinTr {VERSION}</sup>', x=0.5),
        height=500, width=1400,
        showlegend=False,
        margin=dict(l=0, r=0, t=60, b=0)
    )

    fig.show()

    if GUARDAR_HTML:
        html_path = vis_dir / f'{idx_viz:03d}_{nombre}_cd{cd:.4f}.html'
        fig.write_html(str(html_path))

print(f'\nHecho. HTMLs en: {vis_dir}')

In [ ]:
# ── CELDA 7 (extra): Overlay roto+prediccion+GT en un solo panel
# Útil para ver de golpe cómo el modelo completa la pieza rota.

import plotly.graph_objects as go
import numpy as np, torch, random
from pathlib import Path

# Tomar una sola muestra (cambia el índice para ver otra)
IDX = 0
ruta_r, ruta_c = pares_viz[IDX]
nombre = Path(ruta_r).stem.replace('_roto','')

roto_np = np.load(ruta_r).astype(np.float32)
comp_np  = np.load(ruta_c).astype(np.float32)

with torch.no_grad():
    inp = torch.tensor(roto_np).unsqueeze(0).to(device)
    out = model(inp)
    pred_np = (out[-1] if isinstance(out,(list,tuple)) else out).squeeze(0).cpu().numpy()

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=comp_np[:,0], y=comp_np[:,1], z=comp_np[:,2],
    mode='markers', name='GT completo',
    marker=dict(size=1.5, color='#1565C0', opacity=0.25)))
fig.add_trace(go.Scatter3d(
    x=pred_np[:,0], y=pred_np[:,1], z=pred_np[:,2],
    mode='markers', name='Predicción',
    marker=dict(size=2, color='#2E7D32', opacity=0.7)))
fig.add_trace(go.Scatter3d(
    x=roto_np[:,0], y=roto_np[:,1], z=roto_np[:,2],
    mode='markers', name='Roto (entrada)',
    marker=dict(size=2.5, color='#C62828', opacity=0.9)))

cd  = chamfer_np(pred_np, comp_np)
fs  = fscore_np(pred_np, comp_np)
fig.update_layout(
    title=f'Overlay: {nombre} | CD={cd:.4f} | F={fs:.4f} | Modelo: {VERSION}',
    height=700, width=700,
    scene=dict(xaxis=dict(range=[-1.2,1.2],showticklabels=False),
               yaxis=dict(range=[-1.2,1.2],showticklabels=False),
               zaxis=dict(range=[-1.2,1.2],showticklabels=False),
               aspectmode='cube')
)
fig.show()
print('Rojo=entrada rota | Verde=predicción | Azul(transparente)=GT completo')
print(f'Cambia IDX (0..{len(pares_viz)-1}) para ver otra muestra')